# Getting Started with the IRT Python Module

This notebook is a gentle, step-by-step introduction. You will:

- Understand the expected data format
- Fit a Rasch model (MML-EM)
- Inspect item and person reports
- Score people using EAP and MAP

If you are brand new to IRT, just run the cells in order and read the notes between them.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from irt import fit

np.random.seed(123)

## 1) Data format (very important)

Your response matrix must be:

- Rows = people
- Columns = items
- Values = **0**, **1**, or **NaN** (missing)

Below we *simulate* a small dataset just so you can see the workflow. Later notebooks use real data from the R `ltm` and `mirt` packages.

In [3]:
# Simulate a tiny dataset for learning
N = 200  # people
J = 10   # items

theta = np.random.normal(0, 1, size=N)
b = np.linspace(-1.5, 1.5, J)
a = np.ones(J)

# 2PL probability with a=1 is Rasch
p = 1 / (1 + np.exp(-a * (theta[:, None] - b[None, :])))
X = (np.random.rand(N, J) < p).astype(float)

# Add 5% missing values
missing_mask = np.random.rand(N, J) < 0.05
X[missing_mask] = np.nan

item_names = [f"Item {i+1}" for i in range(J)]
person_names = [f"P{i+1:03d}" for i in range(N)]

X_df = pd.DataFrame(X, columns=item_names, index=person_names)
X_df.head()

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10
P001,1.0,0.0,0.0,1.0,0.0,NaN,0.0,0.0,1.0,0.0
P002,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
P003,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,NaN,1.0
P004,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
P005,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,NaN


## 2) Fit a Rasch model (MML-EM)

Rasch = 1-parameter logistic (discrimination fixed at 1). The default estimator is **MML-EM**, which is a standard and stable choice.

In [4]:
result = fit(X_df, model="rasch", estimator="mml_em")
result

FitResult(model='rasch', estimator='mml_em', n_items=10, n_persons=200, n_iter=5, converged, loglik=-1120.78)

## 3) Inspect item and person summaries

These helpers are designed for beginners: you get clean tables without extra work.

In [5]:
item_report = result.item_report()
person_report = result.person_report()

item_report.head(), person_report.head()

(     item    a         b  n_obs
 0  Item 1  1.0 -1.517199    189
 1  Item 2  1.0 -1.110477    192
 2  Item 3  1.0 -0.798887    190
 3  Item 4  1.0 -0.554322    182
 4  Item 5  1.0  0.041754    188,
   person  n_items  sum_score     theta        se
 0   P001        9        3.0 -0.563067  0.611725
 1   P002       10        5.0 -0.013600  0.579910
 2   P003        9        6.0  0.451984  0.610133
 3   P004       10        3.0 -0.695167  0.591882
 4   P005        9        6.0  0.415405  0.605343)

## 4) Score people (EAP and MAP)

- **EAP** = expected a posteriori (posterior mean)
- **MAP** = maximum a posteriori (posterior mode)

Both are standard in `mirt` and `ltm`, and both are supported here.

In [6]:
scores_eap = result.score(method="eap")
scores_map = result.score(method="map")

scores_eap.to_dataframe().head()

,person,theta,se
0,P001,-0.563067,0.611725
1,P002,-0.013600,0.579910
2,P003,0.451984,0.610133
3,P004,-0.695167,0.591882
4,P005,0.415405,0.605343


## 5) Score new people using the same items

If you have new respondents with the **same items**, just pass their responses to `score()`.

In [7]:
new_people = X_df.sample(5, random_state=7)
new_scores = result.score(new_people, method="eap")
new_scores.to_dataframe()

,person,theta,se
0,P087,-0.013600,0.579910
1,P121,-0.013600,0.579910
2,P023,-0.350911,0.582690
3,P012,0.164768,0.605621
4,P196,-1.438546,0.632498


## Next steps

- `01_rasch_ltm_style.ipynb` shows the same workflow using the **LSAT** dataset from R `ltm`
- `02_mirt_style_2pl.ipynb` shows **Rasch + 2PL** workflows aligned with R `mirt`

Those notebooks include side-by-side comparisons with the reference outputs from R.